# Manipulación Avanzada

In [1]:
import pandas as pd

# Tabla 1: Registro de empleados y sus salarios
datos_empleados = {
    'id_empleado': [101, 102, 103, 104, 105, 106],
    'nombre': ['Ana', 'Carlos', 'Luis', 'Marta', 'Sofía', 'Jorge'],
    'departamento': ['IT', 'Ventas', 'IT', 'Marketing', 'Ventas', 'IT'],
    'salario': [3500, 2800, 4000, 3200, 2900, 3800]
}
df_empleados = pd.DataFrame(datos_empleados)

# Tabla 2: Información logística de los departamentos
datos_deps = {
    'departamento': ['IT', 'Ventas', 'Marketing', 'Finanzas'],
    'ciudad': ['Madrid', 'Barcelona', 'Madrid', 'Valencia']
}
df_departamentos = pd.DataFrame(datos_deps)

# Mostramos las estructuras iniciales
print("--- DataFrame: Empleados ---")
print(df_empleados)
print("\n--- DataFrame: Departamentos ---")
print(df_departamentos)

--- DataFrame: Empleados ---
   id_empleado  nombre departamento  salario
0          101     Ana           IT     3500
1          102  Carlos       Ventas     2800
2          103    Luis           IT     4000
3          104   Marta    Marketing     3200
4          105   Sofía       Ventas     2900
5          106   Jorge           IT     3800

--- DataFrame: Departamentos ---
  departamento     ciudad
0           IT     Madrid
1       Ventas  Barcelona
2    Marketing     Madrid
3     Finanzas   Valencia


### Agrupaciones ( .groupby() )
Split (Dividir): El sistema separa las filas del DataFrame en grupos basados en una variable categórica (por ejemplo, junta a todos los de 'IT' por un lado, a los de 'Ventas' por otro).

Apply (Aplicar): Ejecuta una función matemática agregadora sobre los datos numéricos de cada grupo (como una suma, un promedio o un conteo).

Combine (Combinar): Junta los resultados de las operaciones en una nueva tabla resumen compacta.

In [3]:
salario_promedio_dep = df_empleados.groupby('departamento')['salario'].mean()
salario_promedio_dep

departamento
IT           3766.666667
Marketing    3200.000000
Ventas       2850.000000
Name: salario, dtype: float64

### Unir Tablas ( pd.merge() ) Inner Join

Para ello, ambas tablas deben tener una columna cada una que sirva como puente,
en este caso será la columna de departamentos en ambos casos que nos permitirá asignarle a cada
empleado su localización.

In [4]:
df_combinado = pd.merge(df_empleados, df_departamentos)
df_combinado

,id_empleado,nombre,departamento,salario,ciudad
0,101,Ana,IT,3500,Madrid
1,102,Carlos,Ventas,2800,Barcelona
2,103,Luis,IT,4000,Madrid
3,104,Marta,Marketing,3200,Madrid
4,105,Sofía,Ventas,2900,Barcelona
5,106,Jorge,IT,3800,Madrid


In [6]:
df_completo = pd.merge(df_empleados, df_departamentos, how='right')
df_completo

,id_empleado,nombre,departamento,salario,ciudad
0,101.0,Ana,IT,3500.0,Madrid
1,103.0,Luis,IT,4000.0,Madrid
2,106.0,Jorge,IT,3800.0,Madrid
3,102.0,Carlos,Ventas,2800.0,Barcelona
4,105.0,Sofía,Ventas,2900.0,Barcelona
5,104.0,Marta,Marketing,3200.0,Madrid
6,NaN,NaN,Finanzas,NaN,Valencia


Salario promedio por ciudad

In [8]:
promedio_ciudad = df_combinado.groupby('ciudad')['salario'].mean()
promedio_ciudad

ciudad
Barcelona    2850.0
Madrid       3625.0
Name: salario, dtype: float64

#### Uso de múltiples métricas ( .agg(['mean', 'count', 'min']) )

In [9]:
Media_total_min = df_combinado.groupby('ciudad')['salario'].agg(['mean', 'min', 'count'])
Media_total_min

,mean,min,count
ciudad,,,
Barcelona,2850.0,2800,2
Madrid,3625.0,3200,4


Cambio de nombre a la vez que se hace .agg()

In [15]:
df_combinado.groupby('ciudad').agg(salario_promedio = ('salario', 'mean'), salario_mínimo = ('salario', 'min'), tota_empleados = ('salario', 'count')).sort_values(by = 'salario_promedio', ascending= False)

,salario_promedio,salario_mínimo,tota_empleados
ciudad,,,
Madrid,3625.0,3200,4
Barcelona,2850.0,2800,2


## Creamos un nuevo dataset con 1000 registros

In [27]:
import numpy as np

np.random.seed(42) # Garantiza que tengamos los mismos números
n = 1000

df_grande = pd.DataFrame({
    'id_empleado': range(101, 101 + n),
    'nombre': [f"Empleado_{i}" for i in range(1, n + 1)],
    'departamento': np.random.choice(['IT', 'Ventas', 'Marketing', 'Finanzas', 'RRHH', 'Soporte'], n),
    'salario': np.random.randint(2000, 6500, n),
    'experiencia_anios': np.random.randint(1, 16, n)
})

print(f"Dataset inicializado con {len(df_grande)} registros.")

Dataset inicializado con 1000 registros.


### Función .filter() por grupos

In [ ]:
df_filtrado = df_grande.groupby('departamento').filter(lambda x: x['salario'].mean() > 4300) 
#Lambda x toma como x un sub-dataframe con cada categoría de la variable 'departamento' entonces va discriminando estas variables
df_filtrado

,id_empleado,nombre,departamento,salario,experiencia_anios
0,101,Empleado_1,Finanzas,4709,9
10,111,Empleado_11,Finanzas,3160,7
15,116,Empleado_16,Finanzas,4595,13
19,120,Empleado_20,Finanzas,6170,4
22,123,Empleado_23,Finanzas,2117,7
...,...,...,...,...,...
971,1072,Empleado_972,Finanzas,2042,10
973,1074,Empleado_974,Finanzas,4390,5
979,1080,Empleado_980,Finanzas,3014,2
983,1084,Empleado_984,Finanzas,2714,6


In [29]:
df_filtrado['departamento'].unique()

array(['Finanzas'], dtype=object)

In [30]:
print("Registros en df_grande:", len(df_grande))
print("Registros en df_filtrado:", len(df_filtrado))

Registros en df_grande: 1000
Registros en df_filtrado: 174


### Guardar el DataFrame filtrado en csv

In [31]:
df_filtrado.to_csv('empleados_filtrados.csv', index = False)